# Olist E-Commerce Analytics: I. Initial analysis, cleaning, and data structuring.

## Overview

In this part, an analysis for each of the selected 5 tables from the Olist Brazilian E-Commerce database is performed to understand their structure. We want to show the tables to gain the initial illustration of the information available while also accessing the shape, column types, null counts, and duplicate counts. This must be done for further work with the dataset. 

## Pipeline Structure
1. **Environment Setup & Raw Data Ingestion:** Establishing reproducible paths.
2. **Dimensional Processing (Customers):** Geographic string standardisation and zip code formatting.
3. **Fact Table Processing (Orders & Items):** Monitored datetime conversions with active missing-data checks.
4. **Dimensional Processing (Products):** Flattening Portuguese-to-English translations and isolating metadata gaps.
5. **Serialisation:** Compiling the cleaned pipeline into an analysis-ready SQLite Database (`olist_ecommerce.db`).

In [30]:
import pandas as pd
import numpy as np
import os

# Pandas options for clear outputs 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

### Table: Customers Dataset (`olist_customers_dataset.csv`)

In [31]:
# The customers dataset
cust_df = pd.read_csv('data/olist_customers_dataset.csv') # csv load
print(f"Customers dataset shape: {cust_df.shape[0]} rows, {cust_df.shape[1]} columns") # the shape and row count
cust_df.head(5)

Customers dataset shape: 99441 rows, 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [32]:
# The schema contains information and data types for each column
print("Customers schema & data types:")
cust_df.info()

Customers schema & data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


Here it is important to note that we have two columns containing the customer IDs. The first one is `customer_id` and the second is `customer_unique_id`. The former one is used as a key to the `olist_orders_dataset.csv`, thus the primary key for a unique identification of the customer for each order is `customer_id`. However, `customer_unique_id` is the identification of a unique customer on a platform is thus required, so lifetime values and repeat metrics must rely on it. 

In [33]:
# Null count and percentage by column
cust_nulls = pd.DataFrame({
    'Null Count': cust_df.isnull().sum(),
    'Null Percentage (%)': (cust_df.isnull().sum() / len(cust_df)) * 100
}).sort_values(by='Null Count', ascending=False)
display(cust_nulls)

,Null Count,Null Percentage (%)
customer_id,0,0.000
customer_unique_id,0,0.000
customer_zip_code_prefix,0,0.000
customer_city,0,0.000
customer_state,0,0.000


In [34]:
# Primary key duplicate check
pk_col = 'customer_unique_id'
is_unique = cust_df[pk_col].is_unique
dup_count = cust_df[pk_col].duplicated().sum()
n_rows = cust_df.shape[0]
print(f"Primary key: '{pk_col}'")
print(f"Is '{pk_col}' unique? {is_unique}")
print(f"Duplicate count: {dup_count}")
print(f"Duplicate %: {round((dup_count / n_rows)*100, 2)}")
if not is_unique:
    print("\nSample duplicates:")
    display(cust_df[cust_df[pk_col].duplicated(keep=False)].sort_values(by=pk_col).head(6)) # we display the duplicates if they have been found

Primary key: 'customer_unique_id'
Is 'customer_unique_id' unique? False
Duplicate count: 3345
Duplicate %: 3.36

Sample duplicates:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
20023,1b4a75b3478138e99902678254b260f4,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
22066,f6efe5d5c7b85e12355f9d5c3db46da2,004288347e5e88a27ded2bb23747066c,26220,nova iguacu,RJ
72451,49cf243e0d353cd418ca77868e24a670,004b45ec5c64187465168251cd1c9c2f,57055,maceio,AL
87012,d95f60d70d9ea9a7fe37c53c931940bb,004b45ec5c64187465168251cd1c9c2f,57035,maceio,AL


In [35]:
# Primary key duplicate check
pk_col = 'customer_id'
is_unique = cust_df[pk_col].is_unique
dup_count = cust_df[pk_col].duplicated().sum()

print(f"Primary key: '{pk_col}'")
print(f"Is '{pk_col}' unique? {is_unique}")
print(f"Duplicate count: {dup_count}")
if not is_unique:
    print("\nSample duplicates:")
    display(cust_df[cust_df[pk_col].duplicated(keep=False)].sort_values(by=pk_col).head(6)) # we display the duplicates if they have been found

Primary key: 'customer_id'
Is 'customer_id' unique? True
Duplicate count: 0


#### Initial observations:

- The `customer_id` column is fully unique (0 duplicates, as shown above); thus, it may be logically assumed to be the unique identifier for each customer order. 

- `customer_unique_id` column contains duplicate entries, which aligns with how `customer_id` is defined. The existence of duplicates here (about 3.4%) likely indicates the fact that the same customers do repeated purchases. However, the value is rather small, which may indicate that Olist has a very low repeat-customer rate and has to be noted for further analysis. 

- The dataset is 100% complete with zero missing values (0.00% nulls across all columns).

- The `customer_zip_code_prefix` is currently stored as an int64. This is problematic; leading zeros in zip codes will be dropped. The columns to be converted into the `object` and zero-padded to 5 digits. 

- City names (e.g., "sao paulo") may require standardising lowercase letters for clean outputs and aggregations. Accentuation, or special characters, may also be important in specific cases but require more demanding handling. 

### Table: Orders Dataset (`olist_orders_dataset.csv`)

In [36]:
# Load orders dataset
orders_df = pd.read_csv('data/olist_orders_dataset.csv')
print(f"Orders dataset shape: {orders_df.shape[0]} rows, {orders_df.shape[1]} columns")
orders_df.head(3)

Orders dataset shape: 99441 rows, 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [37]:
# Print schema information and dtypes
print("Orders schema & data types:")
orders_df.info()

Orders schema & data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [38]:
# Null count and percentage by column
orders_nulls = pd.DataFrame({
    'Null Count': orders_df.isnull().sum(),
    'Null Percentage (%)': (orders_df.isnull().sum() / len(orders_df)) * 100
}).sort_values(by='Null Count', ascending=False)
display(orders_nulls)

,Null Count,Null Percentage (%)
order_delivered_customer_date,2965,2.982
order_delivered_carrier_date,1783,1.793
order_approved_at,160,0.161
order_id,0,0.000
order_purchase_timestamp,0,0.000
order_status,0,0.000
customer_id,0,0.000
order_estimated_delivery_date,0,0.000


In [39]:
# Primary key duplicate check
pk_col = 'order_id'
is_unique = orders_df[pk_col].is_unique
dup_count = orders_df[pk_col].duplicated().sum()
n_rows = orders_df.shape[0]
print(f"Primary key: '{pk_col}'")
print(f"Is '{pk_col}' unique? {is_unique}")
print(f"Duplicate count: {dup_count}")
print(f"Duplicate %: {round((dup_count / n_rows)*100, 2)}")
if not is_unique: 
    print("\nSample duplicates:")
    display(orders_df[orders_df[pk_col].duplicated(keep=False)].sort_values(by=pk_col).head(6)) # we display the duplicates if they have been found

Primary key: 'order_id'
Is 'order_id' unique? True
Duplicate count: 0
Duplicate %: 0.0


#### Initial observations:




- The primary key `order_id` is fully unique (0 duplicate entries).

- Datetime columns have some null counts that logically correspond to the order processing steps:

  - `order_delivered_customer_date` has `~3.0%` nulls. These orders may have been cancelled, not delivered or changed during shipping or processing.
  
  - `order_delivered_carrier_date` has `~1.8%` nulls. These orders may have not reached the carrier for similar or other reasons.
  - `order_approved_at` has `~0.16%` nulls. Some orders may not have been approved; reasons may differ (e.g., payment gateway rejections).
  - Note: The nulls may not indicate an operating fail case but may rather be a technical problem (e.g., not entered). Thus, for the validation, we would need to use `order_status = 'delivered'` status, and it also makes sense to explore which part of them is actually a technical failure. 

- All 5 temporal columns (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`) are currently stored as strings/objects. For time-series analysis and proper applications for analysis later (e.g., duration calculations), they must be converted into the date/time format (e.g., using `pd.to_datetime()`).

### Table: Order Items Dataset (`olist_order_items_dataset.csv`)

In [40]:
# Load order items dataset
items_df = pd.read_csv('data/olist_order_items_dataset.csv')
print(f"Order items dataset shape: {items_df.shape[0]} rows, {items_df.shape[1]} columns")
items_df.head(3)

Order items dataset shape: 112650 rows, 7 columns


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.900,13.290
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.900,19.930
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.000,17.870


In [41]:
# Print schema information and dtypes
print("Order items schema & data types:")
items_df.info()

Order items schema & data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [42]:
# Due to the fact that we have financial data here it is crucial to check on whether there are negative numbers, outliers or similar problematic cases
print("Financial columns summary:")
display(items_df[['price', 'freight_value']].describe())

Financial columns summary:


,price,freight_value
count,112650.000,112650.000
mean,120.654,19.990
std,183.634,15.806
min,0.850,0.000
25%,39.900,13.080
50%,74.990,16.260
75%,134.900,21.150
max,6735.000,409.680


Here, we observe that the max value here is particularly big, as the standard deviation is only about 183.6. This has to be observed further; this is likely an expensive good, as a lot of various products are sold, yet a mistake may be the case.

In [43]:
# Null count and percentage by column
items_nulls = pd.DataFrame({
    'Null Count': items_df.isnull().sum(),
    'Null Percentage (%)': (items_df.isnull().sum() / len(items_df)) * 100
}).sort_values(by='Null Count', ascending=False)
display(items_nulls)

,Null Count,Null Percentage (%)
order_id,0,0.000
order_item_id,0,0.000
product_id,0,0.000
seller_id,0,0.000
shipping_limit_date,0,0.000
price,0,0.000
freight_value,0,0.000


In [44]:
# Primary key duplicate check, but with the composite key too in this case. 
# This is done due to the fact that each order may contain more than one item and will thus not be unique. 
pk_cols = ['order_id', 'order_item_id']
is_unique = not items_df.duplicated(subset=pk_cols).any()
dup_count = items_df.duplicated(subset=pk_cols).sum()

print(f"Composite primary key candidate: {pk_cols}")
print(f"Is the composite key unique? {is_unique}")
print(f"Duplicate count based on composite key: {dup_count}")

single_pk_dup = items_df['order_id'].duplicated().sum()
dup_percentage = (single_pk_dup / len(items_df)) * 100
print(f"Duplicate count based on 'order_id' alone: {single_pk_dup}")
print(f"Duplicate count % based on 'order_id' alone: {round(dup_percentage, 2)}")

Composite primary key candidate: ['order_id', 'order_item_id']
Is the composite key unique? True
Duplicate count based on composite key: 0
Duplicate count based on 'order_id' alone: 13984
Duplicate count % based on 'order_id' alone: 12.41


In [45]:
# However, to see the percentage of the orders with more than one item, we need a mathematically valid calculation.

# Group the table by order_id to count how many items exist per unique order
items_per_order = items_df.groupby('order_id').size()

# Count how many of those unique orders have a size greater than 1
multi_item_orders_count = (items_per_order > 1).sum()

# Get the absolute total of unique orders in this table
total_unique_orders_in_items = items_df['order_id'].nunique()

# 4. Calculate the true business percentage
true_multi_item_pct = (multi_item_orders_count / total_unique_orders_in_items) * 100

print(f"Total unique orders with purchases: {total_unique_orders_in_items}")
print(f"Orders containing >1 item: {multi_item_orders_count}")
print(f"Percentage of multi-item orders: {round(true_multi_item_pct, 2)}%")

Total unique orders with purchases: 98666
Orders containing >1 item: 9803
Percentage of multi-item orders: 9.94%


About 9.94% of the orders contain more than one item, which should also be noted as it can be useful further.

#### Initial observations:




- A single `order_id` is not unique in this dataset because an order can contain multiple separate items. Combining `['order_id', 'order_item_id']`forms a composite primary key which is 100% unique (0 duplicate entries).

- The dataset is clean with 0% null values across all columns.

- Both `price` and `freight_value` are appropriately stored as float64 dtypes.

- The `shipping_limit_date` is now stored as an object and requires conversion to datetime format.

- For financial data: `price, freight_value`no negative values or $0 prices exist in the dataset (min price: $0.85). 
    - Minimum freight is $0.00, which logically represents valid free-shipping promotions.
    - It has to be noted that the `price` distribution is right-skewed, which requires further exploration. 

### Table: Products Dataset (`olist_products_dataset.csv`)

In [46]:
# Load products dataset
prod_df = pd.read_csv('data/olist_products_dataset.csv')
print(f"Products dataset shape: {prod_df.shape[0]} rows, {prod_df.shape[1]} columns")
prod_df.head(3)

Products dataset shape: 32951 rows, 9 columns


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.000,287.000,1.000,225.000,16.000,10.000,14.000
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.000,276.000,1.000,1000.000,30.000,18.000,20.000
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.000,250.000,1.000,154.000,18.000,9.000,15.000


`product_category_name` here is also formatted to all lowercase with `_` as separators, so for a clear illustration, it should be handled. 

In [47]:
# Print schema information and dtypes
print("Products schema & data types:")
prod_df.info()

Products schema & data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [48]:
# Null count and percentage by column
prod_nulls = pd.DataFrame({
    'Null Count': prod_df.isnull().sum(),
    'Null Percentage (%)': (prod_df.isnull().sum() / len(prod_df)) * 100
}).sort_values(by='Null Count', ascending=False)
display(prod_nulls)

,Null Count,Null Percentage (%)
product_category_name,610,1.851
product_description_lenght,610,1.851
product_name_lenght,610,1.851
product_photos_qty,610,1.851
product_weight_g,2,0.006
product_height_cm,2,0.006
product_length_cm,2,0.006
product_width_cm,2,0.006
product_id,0,0.000


In [49]:
# Primary key duplicate check
pk_col = 'product_id'
is_unique = prod_df[pk_col].is_unique
dup_count = prod_df[pk_col].duplicated().sum()
n_rows = prod_df.shape[0]
print(f"Primary key: '{pk_col}'")
print(f"Is '{pk_col}' unique? {is_unique}")
print(f"Duplicate count: {dup_count}")
print(f"Duplicate %: {round((dup_count / n_rows)*100, 2)}")
if not is_unique: 
    print("\nSample duplicates:")
    display(prod_df[prod_df[pk_col].duplicated(keep=False)].sort_values(by=pk_col).head(6)) # we display the duplicates if they have been found

Primary key: 'product_id'
Is 'product_id' unique? True
Duplicate count: 0
Duplicate %: 0.0


#### Initial observations:




- The primary key `product_id` is 100% unique (0 duplicate entries).

- There is a moderate amount of missing values in product descriptions and descriptive properties:
  - `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` all have 610 missing values (`~1.85%` nulls).
  - Since the absence of a description or photo mathematically equates to zero, these NULL values should be systematically imputed with 0 to ensure correctness in case any calculations are done with it (e.g., cloud storage management).

  - `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm` have 2 missing values (which is `~0.01%` nulls). We may, however, impute them with a median, which is a robust way to deal with it initially (of course, for specific tasks, another approach may be needed).

- Products with missing categories can be mapped to an `'unknown'` category to prevent them from being dropped when join operations are applied.

- We should also note the spelling errors in `product_name_lenght` which we should fix.

### Table: Category Translations (`product_category_name_translation.csv`)

In [50]:
# Load category name translation dataset
trans_df = pd.read_csv('data/product_category_name_translation.csv')
print(f"Category translation shape: {trans_df.shape[0]} rows, {trans_df.shape[1]} columns")
trans_df.head(3)

Category translation shape: 71 rows, 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [51]:
# Print schema information and dtypes
print("Category translation schema & data types:")
trans_df.info()

Category translation schema & data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [52]:
# Null count and percentage by column
trans_nulls = pd.DataFrame({
    'Null Count': trans_df.isnull().sum(),
    'Null Percentage (%)': (trans_df.isnull().sum() / len(trans_df)) * 100
}).sort_values(by='Null Count', ascending=False)
display(trans_nulls)

,Null Count,Null Percentage (%)
product_category_name,0,0.000
product_category_name_english,0,0.000


In [53]:
# Primary key duplicate check
pk_col = 'product_category_name'
is_unique = trans_df[pk_col].is_unique
dup_count = trans_df[pk_col].duplicated().sum()

print(f"Primary key candidate: '{pk_col}'")
print(f"Is '{pk_col}' unique? {is_unique}")
print(f"Duplicate count: {dup_count}")
if not is_unique:
    print("\nSample duplicates:")
    display(trans_df[trans_df[pk_col].duplicated(keep=False)].sort_values(by=pk_col).head(6))

Primary key candidate: 'product_category_name'
Is 'product_category_name' unique? True
Duplicate count: 0


As we should use `product_category_name` for mapping the translations, it might be useful to check the information about the `product_category_name` from `olist_products_dataset.csv`. 

In [54]:
prod_df['product_category_name'].describe()

count               32341
unique                 73
top       cama_mesa_banho
freq                 3029
Name: product_category_name, dtype: object

We observe that there are more unique entries than in the translation table, so this has to be further explored (in the case of 2 non-translated names, it may be handled manually). 

In [55]:
# Identifying the categories in products that are not present in translations table 
prod_cats = set(prod_df['product_category_name'].dropna().unique())
trans_cats = set(trans_df['product_category_name'].unique())

missing_cats = prod_cats - trans_cats
print(f"Categories missing translations ({len(missing_cats)}):")
print(missing_cats)

Categories missing translations (2):
{'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


#### Initial observations:




- The primary key `product_category_name` is 100% unique (0 duplicate entries), which may be used as a mapping dictionary.

- 100% complete translation map with 0% missing values.

- While this translation table contains 71 categories, the `olist_products_dataset.csv` contains 73 distinct values (excluding nulls). This means that there are two untranslated Portuguese category names: `'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'`. During joining, these two categories would remain untranslated or turn into nulls, so it should be handled in a proper way.

### Data schema and data problems summary


#### Relational data model




The dataset follows a snowflake-like schema. We have the following primary key (PK) to foreign key (FK) relationships:




- `olist_orders_dataset` (fact table)
    - Joins to `olist_customers_dataset` via `customer_id`
    - Joins to `olist_order_items_dataset` via `order_id`




- `olist_order_items_dataset` (line-item fact table)
    - Joins to `olist_products_dataset` via `product_id`




- `olist_products_dataset` (dimension table)
    - Joins to `product_category_name_translation` via `product_category_name`




- The PK for the customers table is `customer_id` (transactional), while `customer_unique_id` identifies the actual user. The PK for `order_items` is a composite of `[order_id, order_item_id]`.

#### Data quality issues

- Data type problems: 

    - All timestamp columns across the dataset (`order_purchase_timestamp`, `shipping_limit_date`, etc.) are stored as objects (strings). They must be transformed to datetime.

    - `customer_zip_code_prefix` is stored as int64, which truncates leading zeros. It must be converted to a string and padded, as some 0s were deleted during loading. 

- Relational/mapping problems:

    - The `product_category_name_translation` table is missing mapping keys for two Portuguese categories from the Products table. This may cause data leakage during SQL joins, so it has to be handled. 

- Missing values:

    - About 3% of orders lack delivery dates. This relates to the `order_status` (e.g., canceled/processing). It is logical to leave these nulls when analysing anomalies, yet they should be filtered out when calculating revenues.

    - 610 products (1.85%) are missing the categorical metadata. These should be imputed with an 'unknown' & 0s to preserve the correctness of the calculations. For `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm` we impute NULLs with a median.

- Formatting:

    - `customer_city` contains uncapitalised strings. In case aggregation or clear presentation is done, cities should be formatted properly.

    - `product_category_name` also contains uncapitalised letters and a separator.

    - Some of the column names contain spelling errors (e.g., `product_name_lenght`). 

In [ ]:
import pandas as pd
import sqlite3
import os

print("Starting data pipeline...")

# Raw data loading for a full pipeline reproducibility
data_dir = 'data/'
cust_df = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
orders_df = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
items_df = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
prod_df = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
trans_df = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))

# 1. Customers table
# Restore leading zeros to zip codes
cust_df['customer_zip_code_prefix'] = cust_df['customer_zip_code_prefix'].astype(str).str.zfill(5) # we standardise the zip code to 5 characters with 0 in the beginning

# Properly capitalise Portuguese city names, keeping prepositions lowercase.
def format_pt_city(city_name):
    if pd.isna(city_name):
        return city_name
        
    # Lowercase the entire string first, then split into individual words
    words = str(city_name).lower().split()
    prepositions = {'de', 'do', 'da', 'dos', 'das'}
    
    # Capitalide the word if it is not a preposition, or in case it's the very first word
    formatted_words = [
        word.capitalize() if word not in prepositions or i == 0 else word 
        for i, word in enumerate(words)
    ]
    
    return ' '.join(formatted_words)

# Apply the custom function to the pipeline
cust_df['customer_city'] = cust_df['customer_city'].apply(format_pt_city)

# 2. Orders table
time_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# errors='coerce' is used in order to not throw errors with corrupted date strings while not stopping the pipeline.
# A check is added to ensure we track exactly how many rows are overwritten to NaT.
for col in time_cols:
    natural_nulls = orders_df[col].isna().sum() # count nulls before conversion
    orders_df[col] = pd.to_datetime(orders_df[col], errors='coerce')
    coerced_count = orders_df[col].isna().sum() - natural_nulls # check difference
    
    if coerced_count > 0:
        print(f"WARNING: {coerced_count} corrupted strings coerced to NaT in '{col}'.")

print("olist_orders_dataset cleaned")

# 3. Order items table 
natural_nulls_items = items_df['shipping_limit_date'].isna().sum()
items_df['shipping_limit_date'] = pd.to_datetime(items_df['shipping_limit_date'], errors='coerce')
coerced_count_items = items_df['shipping_limit_date'].isna().sum() - natural_nulls_items

if coerced_count_items > 0:
    print(f"WARNING: {coerced_count_items} corrupted strings coerced to NaT in 'shipping_limit_date'.")

print("olist_order_items_dataset cleaned")

Starting data pipeline...
olist_orders_dataset cleaned
olist_order_items_dataset cleaned


For `olist_products_dataset` it is quite convenient to directly merge the translations, as it helps to eliminate additional tables for memory efficiency and ensure our data is a star schema later, which is usually more effective for Power BI and other tools. 

We may also handle translation of the category to ensure clear consistency overall. 

In [57]:
# 4. Cleaning products & flattening translations
# We merge the English translations to the products table on the product_category_name. 
print("olist_products_dataset & product_category_name_translation:")
# Join translations 
prod_df = pd.merge(prod_df, trans_df, on='product_category_name', how='left')

# spelling errors fixed
prod_df = prod_df.rename(columns={
    'product_name_lenght': 'product_name_length', 
    'product_description_lenght': 'product_description_length'
})

# Manually adding dictionary for the 2 known missing Portuguese translations
missing_translations = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'kitchen_and_food_preparators_portables'
}

# Map the manual translations to the English column where applicable
for pt_name, en_name in missing_translations.items():
    mask = prod_df['product_category_name'] == pt_name
    prod_df.loc[mask, 'product_category_name_english'] = en_name

# Handle the 610 completely missing categories
prod_df['product_category_name_english'] = prod_df['product_category_name_english'].fillna('Unknown')

# Handle the 610 missing descriptive metadata values (filling with 0)
metadata_cols = ['product_name_length', 'product_description_length', 'product_photos_qty']
prod_df[metadata_cols] = prod_df[metadata_cols].fillna(0)
# Fill the 2 missing physical dimensions with the median
dimension_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dimension_cols:
    prod_df[col] = prod_df[col].fillna(prod_df[col].median())

# Drop the original Portuguese column to avoid confusion later
prod_df = prod_df.drop(columns=['product_category_name'])
# Rename English column for cleaner syntax
prod_df = prod_df.rename(columns={'product_category_name_english': 'product_category'})

# English categories standardised for representativeness 
prod_df['product_category'] = prod_df['product_category'].str.replace('_', ' ').str.title() # remove underscores and use title case

print("olist_products_dataset & product_category_name_translation merged and cleaned")
prod_df.head(5)

olist_products_dataset & product_category_name_translation:
olist_products_dataset & product_category_name_translation merged and cleaned


,product_id,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,40.000,287.000,1.000,225.000,16.000,10.000,14.000,Perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,44.000,276.000,1.000,1000.000,30.000,18.000,20.000,Art
2,96bd76ec8810374ed1b65e291975717f,46.000,250.000,1.000,154.000,18.000,9.000,15.000,Sports Leisure
3,cef67bcfe19066a932b7673e239eb23d,27.000,261.000,1.000,371.000,26.000,4.000,26.000,Baby
4,9dc1a7de274444849c219cff195d0b71,37.000,402.000,4.000,625.000,20.000,17.000,13.000,Housewares


For working with SQL queries, a small database is set up locally; this will let us work with the clean versions of the tables conveniently later. 

In [58]:

# 5. Database (SQLite)
db_path = 'olist_ecommerce.db'
conn = sqlite3.connect(db_path)

# Export cleaned dataframes to SQL tables
cust_df.to_sql('customers', conn, if_exists='replace', index=False)
orders_df.to_sql('orders', conn, if_exists='replace', index=False)
items_df.to_sql('order_items', conn, if_exists='replace', index=False)
prod_df.to_sql('products', conn, if_exists='replace', index=False)

conn.close()
print(f"Database saved to {db_path}")

Database saved to olist_ecommerce.db
